In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate, SystemMessagePromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate
from langchain.agents import load_tools, initialize_agent, AgentType, create_react_agent, AgentExecutor
from langchain.memory import ConversationBufferMemory
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

import re
import json
import os
from dotenv import load_dotenv

In [ ]:
dotenv_path = os.path.abspath(os.path.join(os.getcwd(), '..', '.env'))
load_dotenv(dotenv_path)

API_KEY = os.getenv("GOOGLE_API_KEY")

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=API_KEY, temperature=0)
memory = ConversationBufferMemory(memory_key='chat_history')
tools = load_tools(['pubmed'], llm=llm)
agent = initialize_agent(tools, llm, agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION, memory=memory, verbose=True)

In [ ]:
system_message_text = f"""
Eres un agente .....
"""

In [ ]:
memory.chat_memory.add_message(SystemMessage(system_message_text))

result = agent.invoke("hola")['output']

In [ ]:
from langchain.agents import initialize_agent, Tool
from langchain.chat_models import ChatOpenAI
from langchain.tools import BaseTool
from langchain.memory import ConversationBufferMemory
import pandas as pd
import fitz  # PyMuPDF para PDFs

# Tool para leer un PDF
class LeerPDFTool(BaseTool):
    name = "leer_pdf"
    description = "Lee información de un PDF sobre un producto"

    def _run(self, file_path: str):
        doc = fitz.open(file_path)
        text = ""
        for page in doc:
            text += page.get_text()
        return text

# Tool para consultar CSVs con factores de emisión
class ConsultarCSVTool(BaseTool):
    name = "consultar_csv"
    description = "Consulta factores de emisión en archivos CSV"

    def _run(self, query: str):
        materiales = pd.read_csv("factores_materiales.csv")
        transporte = pd.read_csv("factores_transporte.csv")
        # Puedes aplicar lógica para filtrar según el query
        resultado = materiales[materiales["material"].str.contains(query, case=False)]
        return resultado.to_dict() if not resultado.empty else "No se encontró información."

# Tool para calcular la huella de carbono (ejemplo simple)
class CalculadoraEmisionesTool(BaseTool):
    name = "calcular_emisiones"
    description = "Calcula las emisiones estimadas usando datos del producto"

    def _run(self, info_producto: str):
        # Aquí haces el parseo del texto y haces cálculos con pandas
        # Ejemplo: parsear que hay 2kg de acero -> buscar en CSV cuánto CO₂ por kg
        return "Emisión estimada: 3.4 kg CO₂e"

# Lista de herramientas
tools = [
    LeerPDFTool(),
    ConsultarCSVTool(),
    CalculadoraEmisionesTool()
]

# LLM (puede ser OpenAI o el que uses)
llm = ChatOpenAI(temperature=0)

# Agente
agent = initialize_agent(
    tools,
    llm,
    agent_type="zero-shot-react-description",
    memory=ConversationBufferMemory(memory_key="chat_history"),
    verbose=True
)

# Uso del agente
respuesta = agent.run("Lee el archivo 'producto.pdf' y dime cuál es su huella de carbono estimada")
print(respuesta)
